# Chess Match Film — Treino + Export ONNX

**Antes de rodar:** Ative a GPU em `Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU`

Passos:
1. Instala dependências
2. Baixa dataset do Roboflow
3. Verifica mapeamento de classes
4. Treina YOLOv8n (~20–30 min com GPU T4)
5. Exporta para ONNX INT8
6. Baixa o arquivo `chess_pieces_yolov8n.onnx`

In [ ]:
# ── CÉLULA 1: Instalar dependências ──────────────────────────────────────
!pip install roboflow ultralytics -q
import torch
print('GPU disponível:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENHUMA — ative a GPU!')

In [ ]:
# ── CÉLULA 2: Baixar dataset do Roboflow ─────────────────────────────────
# Substitua API_KEY pela sua chave (Roboflow → Settings → Roboflow API)
# Substitua WORKSPACE pelo seu workspace slug (aparece na URL do Roboflow)
# Substitua VERSION pelo número da versão que você criou (provavelmente 1)

API_KEY   = "COLE_SUA_API_KEY_AQUI"
WORKSPACE = "COLE_SEU_WORKSPACE_AQUI"   # ex: "guilhermes-workspace-t7dzd"
PROJECT   = "chess-board-detection-bhl1r"
VERSION   = 1

from roboflow import Roboflow
rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")
print('Dataset baixado em:', dataset.location)

In [ ]:
# ── CÉLULA 3: Verificar mapeamento de classes ─────────────────────────────
# IMPORTANTE: copie o output desta célula e envie para atualizar o app
import yaml
with open(f"{dataset.location}/data.yaml") as f:
    info = yaml.safe_load(f)

print("=== MAPEAMENTO DE CLASSES ===")
names = info.get('names', [])
if isinstance(names, list):
    for i, name in enumerate(names):
        print(f"  {i:2d} = {name}")
elif isinstance(names, dict):
    for k, v in sorted(names.items()):
        print(f"  {k:2d} = {v}")

print(f"\nTotal de classes: {info.get('nc', '?')}")
print(f"Localização: {dataset.location}")

In [ ]:
# ── CÉLULA 4: Treinar YOLOv8n ─────────────────────────────────────────────
# yolov8n = nano (menor e mais rápido — ideal para mobile)
# epochs=50 é suficiente para este dataset; aumente para 100 se quiser mais precisão

from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # baixa automaticamente o modelo base pré-treinado

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,           # GPU
    project='chess_model',
    name='run1',
    patience=10,        # para cedo se não melhorar
    save=True,
    val=True,
)

print('\n=== RESULTADO DO TREINO ===')
print(f'mAP@50:   {results.results_dict.get("metrics/mAP50(B)", "?"):.3f}')
print(f'mAP@50-95:{results.results_dict.get("metrics/mAP50-95(B)", "?"):.3f}')

In [ ]:
# ── CÉLULA 5: Exportar para ONNX ─────────────────────────────────────────
import os
from ultralytics import YOLO

best_weights = 'chess_model/run1/weights/best.pt'
print('Carregando modelo treinado:', best_weights)

trained_model = YOLO(best_weights)

# Exporta para ONNX com quantização INT8 (reduz de ~12MB para ~6MB)
export_path = trained_model.export(
    format='onnx',
    imgsz=640,
    int8=True,
    simplify=True,      # simplifica o grafo ONNX
    dynamic=False,      # batch fixo = mais rápido em mobile
)

print('\nONNX exportado para:', export_path)

# Renomeia para o nome que o app espera
import shutil
final_path = 'chess_pieces_yolov8n.onnx'
shutil.copy(export_path, final_path)
size_mb = os.path.getsize(final_path) / 1024 / 1024
print(f'Arquivo final: {final_path} ({size_mb:.1f} MB)')

In [ ]:
# ── CÉLULA 6: Baixar o arquivo .onnx ─────────────────────────────────────
from google.colab import files
files.download('chess_pieces_yolov8n.onnx')
print('Download iniciado! Salve o arquivo como: chess_pieces_yolov8n.onnx')
print('Depois coloque em: assets/models/chess_pieces_yolov8n.onnx no projeto do app')

In [ ]:
# ── CÉLULA EXTRA: Testar o modelo ONNX com uma imagem ────────────────────
# (opcional — para verificar se o modelo funciona antes de colocar no app)
from ultralytics import YOLO
import glob

onnx_model = YOLO('chess_pieces_yolov8n.onnx')

# Pega a primeira imagem de teste do dataset
test_images = glob.glob(f"{dataset.location}/test/images/*.jpg")[:3]
if not test_images:
    test_images = glob.glob(f"{dataset.location}/valid/images/*.jpg")[:3]

for img_path in test_images:
    results = onnx_model(img_path, conf=0.5)
    results[0].show()  # mostra a imagem com detecções
    print(f'Detecções em {img_path}: {len(results[0].boxes)} peças')